In [ ]:
import os
os.makedirs('/root/.kaggle', exist_ok=True)
os.rename('kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 600)

!kaggle datasets download -d alessiocorrado99/animals10
!unzip -q -o animals10.zip -d animals10

FileNotFoundError: [Errno 2] No such file or directory: 'kaggle.json' -> '/root/.kaggle/kaggle.json'

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np
import pathlib

In [ ]:
data_dir = pathlib.Path("animals10/raw-img")
img_size = (160, 160)
batch_size = 32

In [ ]:
# Аугментация
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.15),
])


In [ ]:
# Датасет
train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=img_size,
    batch_size=batch_size
)

val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=img_size,
    batch_size=batch_size
)

class_names_kz = [
    'ит',         # cane
    'жылқы',      # cavallo
    'піл',        # elefante
    'көбелек',    # farfalla
    'тауық',      # gallina
    'мысық',      # gatto
    'сиыр',       # mucca
    'қой',        # pecora
    'өрмекші',    # ragno
    'тиін'        # scoiattolo
]
np.save("class_names_kz.npy", class_names_kz)


AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)


Found 26179 files belonging to 10 classes.
Using 20944 files for training.
Found 26179 files belonging to 10 classes.
Using 5235 files for validation.


In [ ]:
# MobileNetV2 fine-tuning
base_model = tf.keras.applications.MobileNetV2(
    input_shape=img_size + (3,),
    include_top=False,
    weights="imagenet"
)
base_model.trainable = True

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [ ]:
# Тек жоғарғы 50 қабатты ғана үйретеміз
fine_tune_at = 100
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

inputs = tf.keras.Input(shape=img_size + (3,))
x = data_augmentation(inputs)
x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
x = base_model(x, training=True)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.4)(x)
outputs = layers.Dense(len(class_names_kz), activation='softmax')(x)
model = tf.keras.Model(inputs, outputs)

In [ ]:
# Аз learning rate
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
#Epoch санын көбейтіп, дәлдік арттырамыз
history = model.fit(train_ds, validation_data=val_ds, epochs=15)

Epoch 1/15
655/655 ━━━━━━━━━━━━━━━━━━━━ 92s 73ms/step - accuracy: 0.4049 - loss: 1.9157 - val_accuracy: 0.9100 - val_loss: 0.3398
Epoch 2/15
655/655 ━━━━━━━━━━━━━━━━━━━━ 40s 61ms/step - accuracy: 0.8019 - loss: 0.6298 - val_accuracy: 0.9372 - val_loss: 0.2108
Epoch 3/15
655/655 ━━━━━━━━━━━━━━━━━━━━ 40s 61ms/step - accuracy: 0.8534 - loss: 0.4524 - val_accuracy: 0.9482 - val_loss: 0.1797
Epoch 4/15
655/655 ━━━━━━━━━━━━━━━━━━━━ 40s 62ms/step - accuracy: 0.8713 - loss: 0.3894 - val_accuracy: 0.9528 - val_loss: 0.1631
Epoch 5/15
655/655 ━━━━━━━━━━━━━━━━━━━━ 40s 62ms/step - accuracy: 0.8851 - loss: 0.3526 - val_accuracy: 0.9538 - val_loss: 0.1578
Epoch 6/15
655/655 ━━━━━━━━━━━━━━━━━━━━ 40s 60ms/step - accuracy: 0.9006 - loss: 0.3150 - val_accuracy: 0.9551 - val_loss: 0.1503
Epoch 7/15
655/655 ━━━━━━━━━━━━━━━━━━━━ 41s 62ms/step - accuracy: 0.9073 - loss: 0.2918 - val_accuracy: 0.9576 - val_loss: 0.1416
Epoch 8/15
655/655 ━━━━━━━━━━━━━━━━━━━━ 41s 62ms/step - accuracy: 0.9102 - loss: 0.2673 - 

In [ ]:
# Сақтау
model.save("animals_classifier_mobilenetv2.keras")

In [ ]:
%%writefile app.py
import streamlit as st
import tensorflow as tf
from PIL import Image
import numpy as np

# Веб парақ конфигурациясы
st.set_page_config(page_title="Жануарларды тану", layout="centered")

# Модельді жүктеу
@st.cache_resource
def load_model():
    return tf.keras.models.load_model("animals_classifier_mobilenetv2.keras")

model = load_model()

# Қазақша класс атаулары
CLASS_NAMES = np.load("class_names_kz.npy", allow_pickle=True)

# Интерфейс
st.title("🐾 Жануарларды тану")
st.write("Жануардың суретін жүктеп, жүйенің қандай жануар екенін анықтайтынын көріңіз.")

uploaded_file = st.file_uploader("Суретті таңдаңыз...", type=["jpg", "jpeg", "png"])

if uploaded_file is not None:
    try:
        image = Image.open(uploaded_file).convert("RGB")
        st.image(image, caption="Жүктелген сурет", use_container_width=True)

        # Өлшемін өзгерту және модельге беру
        img = image.resize((160, 160))
        img_array = tf.keras.preprocessing.image.img_to_array(img)
        img_array = tf.expand_dims(img_array, 0)

        # Болжам
        predictions = model.predict(img_array, verbose=0)
        score = tf.nn.softmax(predictions[0])
        predicted_class = CLASS_NAMES[np.argmax(score)]
        confidence = 100 * np.max(score)

        # Нәтиже
        st.success(f"Болжам: **{predicted_class}**")
        st.info(f"Сенімділік: **{confidence:.2f}%**")

        # График
        st.bar_chart(dict(zip(CLASS_NAMES, predictions[0])))

    except Exception as e:
        st.error(f"Қате: {e}")


Writing app.py


In [ ]:
!killall streamlit


streamlit: no process found


In [ ]:
!pip install streamlit pyngrok

In [ ]:
from pyngrok import ngrok
import subprocess
import time

# NGROK токен (нақты)
ngrok.set_auth_token("YOUR_NGROK_TOKEN")

# Streamlit іске қосу
process = subprocess.Popen([
    "streamlit", "run", "app.py",
    "--server.port", "8501",
    "--server.headless", "true"
])

time.sleep(5)
public_url = ngrok.connect(8501)
print("🔗 Веб-парақ:", public_url)


🔗 Веб-парақ: NgrokTunnel: "https://883c-34-143-245-135.ngrok-free.app" -> "http://localhost:8501"
